# 🎯 Урок 12 — Шаблон мини-проекта «Предсказатель» (материалы преподавателя)

Рабочий пример полного цикла на Titanic. Ученики получают ученическую версию и заполняют под свой датасет.

**6 разделов проекта:** задача → данные → baseline → Pipeline → оценка → вывод.

> ★ Обязательно: baseline, Pipeline и честная метрика (урок 11).

## 1. Задача
Предсказываем, выжил ли пассажир Titanic (`survived`). Главная метрика — recall для класса «выжил» (не хотим пропускать выживших).

## 2. Данные

In [ ]:
import seaborn as sns
from sklearn.model_selection import train_test_split
df = sns.load_dataset('titanic')
num = ['age','fare','sibsp','parch']; cat = ['sex','pclass','embarked']
X = df[num+cat]; y = df['survived']
X_tr,X_te,y_tr,y_te = train_test_split(X,y,test_size=0.2,random_state=42)
print('Строк:', len(X)); df[num+cat+['survived']].head()

## 3. Baseline
Любая модель должна обыграть простую.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
base = DummyClassifier(strategy='most_frequent').fit(X_tr,y_tr)
print(f'Baseline accuracy: {accuracy_score(y_te, base.predict(X_te)):.0%}')

## 4. Pipeline (обязательно)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

num = ['age','fare','sibsp','parch']; cat = ['sex','pclass','embarked']
prep = ColumnTransformer([
    ('num', Pipeline([('i',SimpleImputer(strategy='median')),('s',StandardScaler())]), num),
    ('cat', Pipeline([('i',SimpleImputer(strategy='most_frequent')),('o',OneHotEncoder(handle_unknown='ignore'))]), cat),
])
model = Pipeline([('prep',prep),('forest',RandomForestClassifier(n_estimators=100,random_state=42))])
model.fit(X_tr, y_tr)
print('Pipeline обучен.')

## 5. Оценка (метрика + сравнение + кросс-валидация)

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report
print(f'Модель accuracy: {accuracy_score(y_te, model.predict(X_te)):.0%}  (baseline {accuracy_score(y_te, base.predict(X_te)):.0%})')
print(classification_report(y_te, model.predict(X_te), target_names=['погиб','выжил']))
cv = cross_val_score(model, X, y, cv=5)
print(f'Кросс-валидация: {cv.mean():.1%} ± {cv.std():.1%}')

## 6. Вывод (пример формулировки)
> Модель предсказывает выживание с точностью ~80% и обыгрывает baseline (~59%). Самые важные признаки — пол и класс каюты. Модель чаще ошибается на пассажирах 3-го класса. Дальше можно добавить признак «размер семьи».